# NPU matrix multiplication

This example defaults to an 8 x 8 systolic array and loads the matching overlay through the repository runtime, tiles logical M/N dimensions over the physical array, and checks every result against NumPy. K must fit the hardware `MAX_K`; the example never bypasses the runtime with direct MMIO or DMA control.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

repository_root = Path.cwd().resolve()
example_root = repository_root
if (repository_root / 'examples' / 'matrix-multiplication' / 'runtime').is_dir():
    example_root = repository_root / 'examples' / 'matrix-multiplication'
elif not (repository_root / 'src').is_dir():
    repository_root = repository_root.parents[1]
if not (repository_root / 'src').is_dir() or not (example_root / 'runtime').is_dir():
    raise RuntimeError('start this notebook from the package, repository root or example directory')
for import_root in (repository_root, example_root):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from runtime.matrix_multiplication import TiledMatrixMultiplier
from src.runtime import load_pynq_runtime

In [ ]:
artifact_dir = example_root / 'artifacts'
if not artifact_dir.is_dir():
    artifact_dir = repository_root / 'build' / 'vivado' / 'npu_matrix_8x8' / 'artifacts'
manifest_path = artifact_dir / 'npu_matrix.manifest.json'
from src.runtime.verify_overlay import verify_artifacts
manifest = verify_artifacts(artifact_dir)
runtime = load_pynq_runtime(artifact_dir / 'npu_matrix.bit')
multiplier = TiledMatrixMultiplier(runtime)
print({
    'source_commit': manifest['source_commit'],
    'vivado_version': manifest['vivado_version'],
    'target_part': manifest['target_part'],
    'physical_limits': (runtime.max_m, runtime.max_n, runtime.max_k),
})

In [ ]:
def reference(a_matrix, b_matrix):
    return (a_matrix.astype(np.int64) @ b_matrix.astype(np.int64)).astype(np.int32)

def run_and_check(label, a_matrix, b_matrix):
    result = multiplier.run(a_matrix, b_matrix, software_timeout=10.0)
    np.testing.assert_array_equal(result.output, reference(a_matrix, b_matrix))
    metrics = result.metrics
    print({
        'case': label,
        'shape': (metrics.m, metrics.k, metrics.n),
        'tile_count': metrics.tile_count,
        'elapsed_seconds': metrics.elapsed_seconds,
        'operation_count': metrics.operation_count,
        'operations_per_second': metrics.operations_per_second,
    })
    return result

In [ ]:
normal_a = ((np.arange(runtime.max_m * 5).reshape(runtime.max_m, 5) % 17) - 8).astype(np.int8)
normal_b = ((np.arange(5 * runtime.max_n).reshape(5, runtime.max_n) % 19) - 9).astype(np.int8)
normal = run_and_check('full physical array', normal_a, normal_b)
assert normal.metrics.tile_count == 1

## Cross-tile multiplication

For the default 8 x 8 array, multiply 9 x 5 by 5 x 9 to produce a 9 x 9 INT32 result in four physical jobs. Dimensions are derived from the loaded hardware, so explicit 2 x 2 builds remain supported.

In [ ]:
matrix_multiplier = TiledMatrixMultiplier(runtime)
non_aligned_a = ((np.arange((runtime.max_m + 1) * 5).reshape(runtime.max_m + 1, 5) % 17) - 8).astype(np.int8)
non_aligned_b = ((np.arange(5 * (runtime.max_n + 1)).reshape(5, runtime.max_n + 1) % 19) - 9).astype(np.int8)
non_aligned = matrix_multiplier.run(
    non_aligned_a, non_aligned_b, software_timeout=10.0
)
expected_output = reference(non_aligned_a, non_aligned_b)
np.testing.assert_array_equal(non_aligned.output, expected_output)
assert non_aligned.output.shape == (runtime.max_m + 1, runtime.max_n + 1)
assert non_aligned.output.dtype == np.int32
assert non_aligned.metrics.tile_count == 4
print({
    'A shape': non_aligned_a.shape,
    'B shape': non_aligned_b.shape,
    'C shape': non_aligned.output.shape,
    'tile_count': non_aligned.metrics.tile_count,
    'result': non_aligned.output.tolist(),
})

In [ ]:
repeated = run_and_check('repeated', -non_aligned_a, non_aligned_b)
np.testing.assert_array_equal(repeated.output, reference(-non_aligned_a, non_aligned_b))
print('PASS: NPU matrix multiplication example')